# 7단계 · 비지도 이상탐지 3종 + 물리 관계 잔차  [외부 데이터 사용 — B층]

5단계의 '과거기준선'(강건 z)은 변수 **하나하나**가 평소 범위를 벗어났는지만 봤습니다.
이번에는 표준 비지도 모델과 **변수 사이의 관계**를 보는 방법을 같은 조건에서 비교합니다.

| 모델 | 무엇을 이상으로 보나 | 참고 |
|---|---|---|
| Isolation Forest | 무작위로 자를 때 **빨리 고립되는** 쇼트 | Liu, Ting, Zhou (2008) |
| PCA 재구성 오차 | 정상 쇼트들의 **주된 패턴으로 다시 그렸을 때** 안 맞는 정도 | |
| 오토인코더 | 신경망으로 **압축했다 복원했을 때**의 오차 | |
| 물리 관계 잔차 | 공정 원리로 묶인 **변수 관계가 깨진** 정도 | 아래 표 |

**공통 조건 (5단계 교훈 반영)**
- 값은 원래 단위, 기준 데이터는 비라벨 **정상 모드** 쇼트 (라벨 쇼트도 값만 포함, 정답은 안 씀)
- 쇼트마다 **그 시각 이전의 최근 300쇼트**로 학습 (10쇼트마다 다시 학습) → 미래를 보지 않음
- 채점은 6단계와 같은 규칙(예산 5%, 10%)

In [ ]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np, pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor
from src.official import *
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

ML = pd.read_csv(RAW / "moldset_labeled.csv"); MLI = ML.set_index("Unnamed: 0")   # [외부]
DATA, BASE = {}, {}
for p in ["CN7", "RG3"]:
    lab = load(p)[0]
    lab["ts"] = pd.to_datetime(lab["order"].map(MLI["TimeStamp"]))
    DATA[p] = lab
    u = pd.read_csv(RAW / f"unlabeled_raw_{p.lower()}.csv"); u["ts"] = pd.to_datetime(u["TimeStamp"])
    BASE[p] = u[u["Barrel_Temperature_1"] > 100].drop_duplicates("ts").sort_values("ts")   # 정상 모드, 시각당 1쇼트
    print(p, "라벨", len(lab), "행 / 기준 후보 정상 쇼트", len(BASE[p]))

### 물리 관계 5가지
사출 원리상 함께 움직여야 하는 변수들을 하나의 값으로 묶었습니다. 정상이라면 이 값들은 거의 일정합니다.

| # | 관계 | 계산 | 원리 |
|---|---|---|---|
| R1 | 충전 시간 × 사출 속도 | log(Filling_Time) + log(Max_Injection_Speed) | 같은 부피를 채우면 빠를수록 짧다 |
| R2 | 계량 시간 × 스크루 RPM ÷ 계량 위치 | log(Plasticizing_Time) + log(Max_Screw_RPM) − log(Plasticizing_Position) | 같은 양을 계량하면 빨리 돌수록 짧다 |
| R3 | 보압 시간 | Injection_Time − Filling_Time | 사출 중 충전 뒤 남는 시간(설정값에 가까움) |
| R4 | 절환압 ÷ 최대 사출압 | Max_Switch_Over_Pressure / Max_Injection_Pressure | 충전 → 보압 전환 시점의 압력 비율 |
| R5 | 나머지 시간 | Cycle − Injection − Plasticizing − Clamp_Close | 냉각·형개 등 (거의 일정) |

> PINN(Raissi 외, 2019)은 유동·열 **방정식을 공간 위에서** 푸는 방법이라, 쇼트마다 숫자 24개뿐인 이 데이터에는 그대로 쓸 수 없습니다.
> 대신 공정 원리를 **특징(관계)**으로 넣은 것이 이 방법입니다.

> **질문 7-1.** CN7 둘다불량 3쇼트는 사출 속도가 평소보다 크게 튀었습니다(z ≈ 9). 그렇다면 5개 관계 중 어느 것이 깨졌을까요?
>
> 내 예상:

In [ ]:
EPS = 1e-6
def physics(X):
    D = pd.DataFrame(X, columns=VARS)
    return np.column_stack([
        np.log(D["Filling_Time"] + EPS) + np.log(D["Max_Injection_Speed"] + EPS),
        np.log(D["Plasticizing_Time"] + EPS) + np.log(D["Max_Screw_RPM"] + EPS) - np.log(D["Plasticizing_Position"] + EPS),
        D["Injection_Time"] - D["Filling_Time"],
        D["Max_Switch_Over_Pressure"] / (D["Max_Injection_Pressure"] + EPS),
        D["Cycle_Time"] - D["Injection_Time"] - D["Plasticizing_Time"] - D["Clamp_Close_Time"],
    ])
PHYS_NAMES = ["R1 충전×속도", "R2 계량", "R3 보압시간", "R4 압력비", "R5 나머지시간"]


class RobustScaler:
    """중앙값·MAD(또는 표준편차 중 큰 쪽)로 표준화. 변하지 않는 변수는 0으로."""
    def fit(self, P):
        self.med = np.median(P, axis=0)
        sd = np.maximum(np.median(np.abs(P - self.med), axis=0) * 1.4826, P.std(axis=0))
        self.ok = sd > 1e-6
        self.sd = np.where(self.ok, sd, 1)
        return self
    def transform(self, X):
        return np.where(self.ok, np.clip((X - self.med) / self.sd, -20, 20), 0)


def make_scorers():
    """(학습 함수, 점수 함수) 묶음. 점수는 클수록 이상."""
    def fit_if(Z):   return IsolationForest(n_estimators=100, random_state=0).fit(Z)
    def sc_if(m, z): return -m.score_samples(z[None])[0]
    def fit_pca(Z):  return PCA(n_components=0.9).fit(Z)
    def sc_pca(m, z): return np.sqrt(((z - m.inverse_transform(m.transform(z[None]))[0]) ** 2).mean())
    def fit_ae(Z):   return MLPRegressor(hidden_layer_sizes=(12, 4, 12), max_iter=300, random_state=0).fit(Z, Z)
    def sc_ae(m, z): return np.sqrt(((z - m.predict(z[None])[0]) ** 2).mean())
    return {"Isolation Forest": (fit_if, sc_if), "PCA 재구성": (fit_pca, sc_pca), "오토인코더": (fit_ae, sc_ae)}


def rolling_scores(lab, base, W=300, refit=10, static=False):
    """쇼트마다 과거 W 쇼트로 학습(refit 쇼트마다 갱신)한 모델들의 점수. static=True 면
    첫 라벨 쇼트 이전의 과거 전체로 한 번만 학습(정적 모델)."""
    first = ~lab["pair"].duplicated()
    X = MLI.loc[lab.loc[first, "order"], VARS].values
    T = lab.loc[first, "ts"].values
    bt, BX = base["ts"].values, base[VARS].values
    scorers = make_scorers()
    out = {k: np.zeros(len(X)) for k in list(scorers) + ["물리 관계 잔차", "강건 z (5단계)"]}
    phys_z = np.zeros((len(X), 5))
    models = None
    for i, (x, t) in enumerate(zip(X, T)):
        if models is None or (not static and i % refit == 0):
            P = BX[bt < T[0]][-3000:] if static else BX[bt < t][-W:]
            sc = RobustScaler().fit(P); Z = sc.transform(P)
            models = {k: f(Z) for k, (f, _) in scorers.items()}
            psc = RobustScaler().fit(physics(P))
        z = sc.transform(x)
        for k, (_, s) in scorers.items():
            out[k][i] = s(models[k], z)
        pz = np.abs(psc.transform(physics(x[None])[0]))
        phys_z[i] = pz
        out["물리 관계 잔차"][i] = np.sqrt((pz ** 2).mean())
        out["강건 z (5단계)"][i] = np.sqrt((np.abs(z) ** 2).mean())
    pairs = lab.loc[first, "pair"].values
    rows = {k: lab["pair"].map(pd.Series(v, index=pairs)).values for k, v in out.items()}
    return rows, pd.DataFrame(phys_z, index=pairs, columns=PHYS_NAMES)

> **질문 7-2.** '최근 300쇼트로 계속 다시 학습(롤링)'과 '라벨 기간 전의 과거 3,000쇼트로 한 번만 학습(정적)' 중 어느 쪽이 나을까요?
> 힌트: 5-5의 (가)가 왜 실패했는지 떠올려 보세요.
>
> 내 예상:

In [ ]:
SC, PHYS = {}, {}
for p in ["CN7", "RG3"]:
    t0 = time.time()
    roll, PHYS[p] = rolling_scores(DATA[p], BASE[p])
    stat, _ = rolling_scores(DATA[p], BASE[p], static=True)
    SC[p] = {**{f"{k} (롤링)": v for k, v in roll.items()},
             **{f"{k} (정적)": v for k, v in stat.items() if k in ["PCA 재구성", "강건 z (5단계)"]}}
    print(p, f"{time.time() - t0:.0f}초")

In [ ]:
def budget_f1(lab, s, q):
    y = lab["y"].values
    o = np.lexsort((np.random.RandomState(0).rand(len(s)), -np.asarray(s, float)))
    k = int(round(len(s) * q)); tp = int(y[o[:k]].sum())
    return tp, 2 * tp / (k + y.sum())

def both_rank(lab, s):
    sh = pd.Series(s, index=lab.index).groupby(lab["pair"]).first().rank(ascending=False, pct=True) * 100
    both = lab.groupby("pair")["y"].transform("sum") == 2
    return sh[lab.loc[both, "pair"].unique()].round(1).tolist()

rows = []
for p, lab in DATA.items():
    P, ceil5 = int(lab["y"].sum()), None
    k5 = int(round(len(lab) * 0.05)); ceil5 = 2 * min(P, k5) / (k5 + P)
    for name, s in SC[p].items():
        e = evaluate(lab, s, name, p, 1)
        tp5, f5 = budget_f1(lab, s, 0.05); tp10, f10 = budget_f1(lab, s, 0.10)
        rows.append({"모델": name, "제품": p, "AP": e["AP"], "유의": e["유의"],
                     "F1@5%": f5, "TP@5%": tp5, "F1@10%": f10, "TP@10%": tp10,
                     "예산5% 상한 대비": f5 / ceil5, "둘다불량 순위%": both_rank(lab, s)})
T7 = pd.DataFrame(rows)
T7.round(3)

### 어떤 물리 관계가 깨졌나 (CN7 둘다불량 3쇼트 vs 나머지)
값은 관계별 |z| (평소 흔들림의 몇 배).

In [ ]:
lab = DATA["CN7"]
kind = lab.groupby("pair")["y"].sum().reindex(PHYS["CN7"].index)
n = lab.groupby("pair").size().reindex(PHYS["CN7"].index)
group = np.where(kind == 2, "둘다불량", np.where(kind == 1, "쌍둥이", "양품"))
print(PHYS["CN7"].loc[kind == 2].round(1).to_string(), "\n")
print("집단별 중앙값"); print(PHYS["CN7"].groupby(group).median().round(2).to_string())

> **질문 7-3.** 표를 보고 답해 보세요.
> 1. 비지도 모델 중 CN7에서 가장 나은 것은? 5단계 강건 z와 비교하면?
> 2. 정적 모델은 왜 그런 결과가 나왔나?
> 3. RG3는?
> 4. 이 결과가 "모델을 바꿔도 정보 상한은 넘지 못한다"는 주장에 어떤 근거가 되나?
>
> 내 답: